# Feature Engineering

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/processed/Airbnb_Data.csv')

In [3]:
df.columns

Index(['log_price', 'property_type', 'room_type', 'amenities', 'accommodates',
       'bathrooms', 'bed_type', 'cancellation_policy', 'cleaning_fee', 'city',
       'host_has_profile_pic', 'host_identity_verified', 'host_response_rate',
       'instant_bookable', 'latitude', 'longitude', 'number_of_reviews',
       'review_scores_rating', 'bedrooms', 'beds'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59463 entries, 0 to 59462
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   log_price               59463 non-null  float64
 1   property_type           59463 non-null  object 
 2   room_type               59463 non-null  object 
 3   amenities               59463 non-null  object 
 4   accommodates            59463 non-null  int64  
 5   bathrooms               59463 non-null  float64
 6   bed_type                59463 non-null  object 
 7   cancellation_policy     59463 non-null  object 
 8   cleaning_fee            59463 non-null  bool   
 9   city                    59463 non-null  object 
 10  host_has_profile_pic    59463 non-null  object 
 11  host_identity_verified  59463 non-null  object 
 12  host_response_rate      59463 non-null  float64
 13  instant_bookable        59463 non-null  bool   
 14  latitude                59463 non-null

## One-Hot Encoding 

In [5]:
df = pd.get_dummies(df, columns=['property_type'], drop_first=True)

In [6]:
df = pd.get_dummies(df, columns=['room_type'], drop_first=True)

In [7]:
df = pd.get_dummies(df, columns=['bed_type'], drop_first=True)

In [8]:
df = pd.get_dummies(df, columns=['cancellation_policy'], drop_first=True)

In [9]:
df = pd.get_dummies(df, columns=['city'], drop_first=True)

In [10]:
df.columns

Index(['log_price', 'amenities', 'accommodates', 'bathrooms', 'cleaning_fee',
       'host_has_profile_pic', 'host_identity_verified', 'host_response_rate',
       'instant_bookable', 'latitude', 'longitude', 'number_of_reviews',
       'review_scores_rating', 'bedrooms', 'beds',
       'property_type_Bed & Breakfast', 'property_type_Boat',
       'property_type_Boutique hotel', 'property_type_Bungalow',
       'property_type_Cabin', 'property_type_Camper/RV',
       'property_type_Casa particular', 'property_type_Castle',
       'property_type_Cave', 'property_type_Chalet',
       'property_type_Condominium', 'property_type_Dorm',
       'property_type_Earth House', 'property_type_Guest suite',
       'property_type_Guesthouse', 'property_type_Hostel',
       'property_type_House', 'property_type_Hut', 'property_type_In-law',
       'property_type_Island', 'property_type_Lighthouse',
       'property_type_Loft', 'property_type_Other',
       'property_type_Parking Space', 'property_ty

## Label Enconding

In [11]:
# Pasar a 0 y 1 
df['host_has_profile_pic'] = df['host_has_profile_pic'].map({'t': 1, 'f': 0, '-1': -1})

In [12]:
df['host_identity_verified'] = df['host_identity_verified'].map({'t': 1, 'f': 0, '-1': -1})

In [13]:
df['cleaning_fee'] = df['cleaning_fee'].map({True: 1, False: 0})

In [14]:
df['instant_bookable'] = df['instant_bookable'].map({True: 1, False: 0})

In [15]:
print(df['host_has_profile_pic'].value_counts())
print(df['host_identity_verified'].value_counts())
print(df['cleaning_fee'].value_counts())
print(df['instant_bookable'].value_counts())

host_has_profile_pic
 1    59117
 0      189
-1      157
Name: count, dtype: int64
host_identity_verified
 1    38690
 0    20616
-1      157
Name: count, dtype: int64
cleaning_fee
1    42417
0    17046
Name: count, dtype: int64
instant_bookable
0    44782
1    14681
Name: count, dtype: int64


## Extracción de datos

In [16]:
df['amenities']

0        {"Wireless Internet","Air conditioning",Kitche...
1        {TV,"Cable TV","Wireless Internet","Air condit...
2        {TV,Internet,"Wireless Internet","Air conditio...
3        {TV,"Wireless Internet",Heating,"Smoke detecto...
4        {TV,Internet,"Wireless Internet","Air conditio...
                               ...                        
59458    {TV,"Cable TV",Internet,"Wireless Internet","A...
59459    {TV,Internet,"Wireless Internet","Air conditio...
59460                                                   {}
59461    {TV,Internet,"Wireless Internet","Air conditio...
59462    {TV,"Wireless Internet","Air conditioning",Kit...
Name: amenities, Length: 59463, dtype: object

In [17]:
import re
from sklearn.preprocessing import MultiLabelBinarizer
def limpiar_amenities(amenities):
    if pd.isna(amenities) or amenities.strip() == "":
        return ""
    
    # Eliminar llaves y comillas
    amenities = re.sub(r'[{}"]', '', amenities)

    # Reemplazar valores extraños
    amenities = re.sub(r'translation missing: en\.(hosting_amenity_\d+)', r'\1', amenities)

    # Eliminar dobles comas
    amenities = re.sub(r',\s*,', ',', amenities).strip(',')

    # Eliminar espacios extra
    amenities = re.sub(r'\s+', ' ', amenities)

    # Eliminar valores vacios
    amenities = ','.join([amenity for amenity in amenities.split(',') if amenity.strip() != ""])

    return amenities

In [18]:
df['clean_amenities'] = df['amenities'].apply(limpiar_amenities)

In [19]:
df['clean_amenities'].unique()

array(['Wireless Internet,Air conditioning,Kitchen,Heating,Family/kid friendly,Essentials,Hair dryer,Iron,hosting_amenity_50',
       'TV,Cable TV,Wireless Internet,Air conditioning,Kitchen,Breakfast,Buzzer/wireless intercom,Heating,Family/kid friendly,Smoke detector,Carbon monoxide detector,Fire extinguisher,Essentials,Shampoo,Hangers,Hair dryer,Iron,Laptop friendly workspace,hosting_amenity_50',
       'TV,Internet,Wireless Internet,Air conditioning,Kitchen,Elevator in building,Heating,Smoke detector,Carbon monoxide detector,Fire extinguisher,Essentials,Shampoo',
       ...,
       'TV,Cable TV,Internet,Wireless Internet,Air conditioning,Wheelchair accessible,Pool,Kitchen,Doorman,Gym,Elevator in building,Heating,Family/kid friendly,Washer,Dryer,Smoke detector,Carbon monoxide detector,First aid kit,Safety card,Fire extinguisher,Essentials,Shampoo,Hangers,Hair dryer,Iron,Laptop friendly workspace',
       'TV,Internet,Wireless Internet,Air conditioning,Kitchen,Free parking on premises,

In [20]:
# Contar las amenities
df["amenities_count"] = df["amenities"].apply(lambda x: len(x.split(',')))

In [21]:
df["amenities_count"]

0         9
1        19
2        12
3        10
4        21
         ..
59458    26
59459    13
59460     1
59461    31
59462    15
Name: amenities_count, Length: 59463, dtype: int64

In [22]:
# Convertir las cadenas de texto en listas de amenities
df['clean_amenities'] = df['clean_amenities'].apply(lambda x: x.split(','))


In [23]:
# Inicializar MultiLabelBinarizer
mlb = MultiLabelBinarizer()

# Aplicar el One-Hot Encoding
amenities_encoded = pd.DataFrame(mlb.fit_transform(df['clean_amenities']), columns=mlb.classes_)
amenities_encoded = amenities_encoded.loc[:, amenities_encoded.columns != '']
amenities_encoded.columns = amenities_encoded.columns.str.strip()

# Combinar los valores de ambas columnas, tomando el valor de una si está presente
amenities_encoded['Wide clearance to shower and toilet'] = amenities_encoded['Wide clearance to shower and toilet'].fillna(amenities_encoded['Wide clearance to shower & toilet'])

# Eliminar la columna redundante
amenities_encoded = amenities_encoded.drop(columns=['Wide clearance to shower & toilet'])

# Concatenar el DataFrame original con las nuevas columnas One-Hot encoded
#df = pd.concat([df, amenities_encoded], axis=1)

In [24]:
amenities_encoded.shape

(59463, 129)

In [25]:
amenities_encoded.columns

Index(['smooth pathway to front door', '24-hour check-in',
       'Accessible-height bed', 'Accessible-height toilet', 'Air conditioning',
       'Air purifier', 'BBQ grill', 'Baby bath', 'Baby monitor',
       'Babysitter recommendations',
       ...
       'Wheelchair accessible', 'Wide clearance to bed',
       'Wide clearance to shower and toilet', 'Wide doorway', 'Wide entryway',
       'Wide hallway clearance', 'Window guards', 'Wireless Internet',
       'hosting_amenity_49', 'hosting_amenity_50'],
      dtype='object', length=129)

In [26]:
# Suma de los amenities 'Accessible-height bed', 'Accessible-height toilet', 'Fixed grab bars for shower & toilet', 'Grab-rails for shower and toilet', 'Roll-in shower with chair', 'Wheelchair accessible', 'Wide clearance to bed', 'Wide clearance to shower and toilet', 'Wide doorway', 'Wide entryway', 'Step-free access', 'Wide hallway clearance'
df['accessible_amenities'] = amenities_encoded[[
    'Accessible-height bed', 'Accessible-height toilet', 'Disabled parking spot',
    'Elevator', 'Elevator in building', 'Fixed grab bars for shower & toilet',
    'Flat smooth pathway to front door', 'Grab-rails for shower and toilet',
    'Ground floor access', 'Roll-in shower with chair', 'Single level home',
    'Step-free access', 'Well-lit path to entrance', 'Wheelchair accessible',
    'Wide clearance to bed', 'Wide clearance to shower and toilet',
    'Wide doorway', 'Wide entryway', 'Wide hallway clearance'
]].sum(axis=1)
amenities = {'accessible_amenities': [
    'Accessible-height bed', 'Accessible-height toilet', 'Disabled parking spot',
    'Elevator', 'Elevator in building', 'Fixed grab bars for shower & toilet',
    'Flat smooth pathway to front door', 'Grab-rails for shower and toilet',
    'Ground floor access', 'Roll-in shower with chair', 'Single level home',
    'Step-free access', 'Well-lit path to entrance', 'Wheelchair accessible',
    'Wide clearance to bed', 'Wide clearance to shower and toilet',
    'Wide doorway', 'Wide entryway', 'Wide hallway clearance'
]}

In [27]:
df['accessible_amenities'].value_counts()

accessible_amenities
0     42951
1     12886
2      3074
3       135
4       104
5        73
7        64
6        63
8        61
10       24
9        22
11        5
12        1
Name: count, dtype: int64

In [28]:
# suma de las amenities 'Bath towel', 'Bathtub', 'Bathtub with shower chair', 'Body soap', 'Shampoo', 'Hand or paper towel', 'Hand soap', 'Toilet paper', 'Fireplace guards'
df['bathroom_amenities'] = amenities_encoded[[
    'Bath towel', 'Bathtub', 'Bathtub with shower chair', 'Body soap',
    'Hand or paper towel', 'Hand soap', 'Handheld shower head',
    'Private bathroom', 'Shampoo', 'Toilet paper', 'Hot tub', 'Dryer'
]].sum(axis=1)
amenities['bathroom_amenities'] = [
    'Bath towel', 'Bathtub', 'Bathtub with shower chair', 'Body soap',
    'Hand or paper towel', 'Hand soap', 'Handheld shower head',
    'Private bathroom', 'Shampoo', 'Toilet paper', 'Hot tub', 'Dryer'
]

In [29]:
df['bathroom_amenities'].value_counts()

bathroom_amenities
1    23063
2    21753
0    10520
3     3940
4      181
5        5
7        1
Name: count, dtype: int64

In [30]:
# suma de amenities 'Air conditioning', 'Heating', 'Air purifier'
df['climate_amenities'] = amenities_encoded[[
    'Air conditioning', 'Air purifier', 'Essentials', 'Extra pillows and blankets',
    'Firm mattress', 'Heating', 'Hot water', 'Room-darkening shades'
]].sum(axis=1)
amenities['climate_amenities'] = [
    'Air conditioning', 'Air purifier', 'Essentials', 'Extra pillows and blankets',
    'Firm mattress', 'Heating', 'Hot water', 'Room-darkening shades'
]

In [31]:
df['climate_amenities'].value_counts()

climate_amenities
3    34191
2    15695
1     4334
4     2007
0     1685
5     1353
6      187
7       11
Name: count, dtype: int64

In [32]:
# Suma de 'Cat(s)', 'Dog(s)', 'Other pet(s)', 'Pets allowed', 'Pets live on this property'
df['pet_amenities'] = amenities_encoded[[
    'Cat(s)', 'Dog(s)', 'Other pet(s)', 'Pets allowed', 'Pets live on this property'
]].sum(axis=1)
amenities['pet_amenities'] = [
    'Cat(s)', 'Dog(s)', 'Other pet(s)', 'Pets allowed', 'Pets live on this property'
]

In [33]:
df['pet_amenities'].value_counts()

pet_amenities
0    45868
1     7214
2     4404
3     1650
4      291
5       36
Name: count, dtype: int64

In [34]:
# suma amenities 'Essentials', 'Internet', 'Wireless Internet', 'Smoke detector', 'Fire extinguisher', 'Shampoo', 'Iron', 'Hair dryer', 'Shower', 'Luggage dropoff allowed', 'Free parking on premises', 'Free parking on street', 'Cable TV', 'Washer / Dryer'
df['basic_amenities'] = amenities_encoded[[
    '24-hour check-in', 'Bed linens', 'Cleaning before checkout', 'Essentials',
    'Hair dryer', 'Hangers', 'Internet', 'Iron', 'Laptop friendly workspace',
    'Lock on bedroom door', 'Private entrance', 'Private living room', 'Safety card',
    'Self Check-In', 'Smart lock', 'Smartlock', 'Smoke detector', 'Wireless Internet',
    'Free parking on premises'
]].sum(axis=1)
amenities['basic_amenities'] = [
    '24-hour check-in', 'Bed linens', 'Cleaning before checkout', 'Essentials',
    'Hair dryer', 'Hangers', 'Internet', 'Iron', 'Laptop friendly workspace',
    'Lock on bedroom door', 'Private entrance', 'Private living room', 'Safety card',
    'Self Check-In', 'Smart lock', 'Smartlock', 'Smoke detector', 'Wireless Internet',
    'Free parking on premises'
]

In [35]:
df['basic_amenities'].value_counts()

basic_amenities
8     7653
7     7315
9     6918
6     6651
4     6538
5     6280
10    5206
3     4196
11    2988
2     2157
12    1393
0      940
1      644
13     433
14     124
15      23
16       4
Name: count, dtype: int64

In [36]:
# suma amenities 'Carbon monoxide detector', 'First aid kit', 'Fireplace guards', 'Smart lock', 'Keypad', 'Lock on bedroom door', 'Smoke detector', 'Safety card', 'Self Check-In'
df['safety_amenities'] = amenities_encoded[[
    'Buzzer/wireless intercom', 'Carbon monoxide detector', 'Doorman',
    'Doorman Entry', 'Fire extinguisher', 'Fireplace guards', 'First aid kit',
    'Host greets you', 'Keypad', 'Lockbox', 'Outlet covers', 'Safety card',
    'Stair gates', 'Table corner guards', 'Window guards'
]].sum(axis=1)
amenities['safety_amenities'] = [
    'Buzzer/wireless intercom', 'Carbon monoxide detector', 'Doorman',
    'Doorman Entry', 'Fire extinguisher', 'Fireplace guards', 'First aid kit',
    'Host greets you', 'Keypad', 'Lockbox', 'Outlet covers', 'Safety card',
    'Stair gates', 'Table corner guards', 'Window guards'
]

In [37]:
df['safety_amenities'].value_counts()

safety_amenities
1     14975
2     13805
0     11588
3      9819
4      6305
5      2348
6       513
7        88
8        15
9         5
10        2
Name: count, dtype: int64

In [38]:
# suma amenities 'BBQ grill', 'Coffee maker', 'Cooking basics', 'Dishwasher', 'Microwave', 'Oven', 'Stove', 'Refrigerator', 'Dishes and silverware', 'Iron', 'Hot water kettle'
df['kitchen_amenities'] = amenities_encoded[[
    'Breakfast', 'Coffee maker', 'Cooking basics', 'Dishes and silverware',
    'Dishwasher', 'Hot water kettle', 'Kitchen', 'Microwave', 'Oven',
    'Pack ’n Play/travel crib', 'Refrigerator', 'Stove'
]].sum(axis=1)
amenities['kitchen_amenities'] = [
    'Breakfast', 'Coffee maker', 'Cooking basics', 'Dishes and silverware',
    'Dishwasher', 'Hot water kettle', 'Kitchen', 'Microwave', 'Oven',
    'Pack ’n Play/travel crib', 'Refrigerator', 'Stove'
]

In [39]:
df['kitchen_amenities'].value_counts()

kitchen_amenities
1     45524
2      5997
0      4441
9       990
8       906
7       616
6       269
10      227
4       169
5       162
3       135
11       26
12        1
Name: count, dtype: int64

In [40]:
# suma amenities 'Garden or backyard', 'Patio or balcony', 'Beachfront', 'Waterfront', 'Lake access', 'Ski in/Ski out'
df['outdoor_amenities'] = amenities_encoded[[
    'BBQ grill', 'Beach essentials', 'Beachfront', 'EV charger',
    'Free parking on premises', 'Free parking on street', 'Garden or backyard',
    'Gym', 'Lake access', 'Patio or balcony', 'Pool', 'Ski in/Ski out',
    'Suitable for events', 'Waterfront', 'Path to entrance lit at night',
]].sum(axis=1)
amenities['outdoor_amenities'] = [
    'BBQ grill', 'Beach essentials', 'Beachfront', 'EV charger',
    'Free parking on premises', 'Free parking on street', 'Garden or backyard',
    'Gym', 'Lake access', 'Patio or balcony', 'Pool', 'Ski in/Ski out',
    'Suitable for events', 'Waterfront', 'Path to entrance lit at night',
]

In [41]:
df['outdoor_amenities'].value_counts()

outdoor_amenities
0    35474
1    17192
2     4526
3     1983
4      240
5       33
6        8
7        7
Name: count, dtype: int64

In [42]:
# suma amenities 'TV', 'Cable TV', 'Game console', 'Smartlock', 'Pocket wifi', 'Laptop friendly workspace', 'Buzzer/wireless intercom', 'Internet'
df['entertainment_amenities'] = amenities_encoded[[
    'Cable TV', 'Ethernet connection', 'Game console', 'Indoor fireplace',
    'Pocket wifi', 'TV', 'Gym', 'Suitable for events'
]].sum(axis=1)
amenities['entertainment_amenities'] = [
    'Cable TV', 'Ethernet connection', 'Game console', 'Indoor fireplace',
    'Pocket wifi', 'TV', 'Gym', 'Suitable for events'
]

In [43]:
df['entertainment_amenities'].value_counts()

entertainment_amenities
1    21452
2    16543
0    15209
3     5284
4      885
5       85
6        5
Name: count, dtype: int64

In [44]:
# suma amenities 'Family/kid friendly', 'Host greets you', 'Luggage dropoff allowed', 'Long term stays allowed', 'Pets live on this property', 'Paid parking off premises', 'Private bathroom', 'Private entrance', 'Private living room'
df['family_amenities'] = amenities_encoded[[
    'Baby bath', 'Baby monitor', 'Babysitter recommendations', 'Changing table',
    'Children’s books and toys', 'Children’s dinnerware', 'Crib',
    'Family/kid friendly', 'High chair'
]].sum(axis=1)
amenities['family_amenities'] = [
    'Baby bath', 'Baby monitor', 'Babysitter recommendations', 'Changing table',
    'Children’s books and toys', 'Children’s dinnerware', 'Crib',
    'Family/kid friendly', 'High chair'
]

In [45]:
# Mira cuantos valores hay en el diccionario amenities
total_amenities = 0
amenities_list = []
for key in amenities:
    total_amenities += len(amenities[key])
    amenities_list += amenities[key]
    print(f'{key}: {len(amenities[key])}')
print(f'Total: {total_amenities}')

accessible_amenities: 19
bathroom_amenities: 12
climate_amenities: 8
pet_amenities: 5
basic_amenities: 19
safety_amenities: 15
kitchen_amenities: 12
outdoor_amenities: 15
entertainment_amenities: 8
family_amenities: 9
Total: 122


In [46]:
len(amenities_list)

122

In [47]:
amenities_columns = [
    "smooth pathway to front door", "24-hour check-in", "Accessible-height bed", "Accessible-height toilet",
    "Air conditioning", "Air purifier", "BBQ grill", "Baby bath", "Baby monitor",
    "Babysitter recommendations", "Bath towel", "Bathtub", "Bathtub with shower chair",
    "Beach essentials", "Beachfront", "Bed linens", "Body soap", "Breakfast",
    "Buzzer/wireless intercom", "Cable TV", "Carbon monoxide detector", "Cat(s)",
    "Changing table", "Children’s books and toys", "Children’s dinnerware",
    "Cleaning before checkout", "Coffee maker", "Cooking basics", "Crib",
    "Disabled parking spot", "Dishes and silverware", "Dishwasher", "Dog(s)", "Doorman",
    "Doorman Entry", "Dryer", "EV charger", "Elevator", "Elevator in building",
    "Essentials", "Ethernet connection", "Extra pillows and blankets",
    "Family/kid friendly", "Fire extinguisher", "Fireplace guards", "Firm matress",
    "Firm mattress", "First aid kit", "Fixed grab bars for shower & toilet", "Flat",
    "Flat smooth pathway to front door", "Free parking on premises",
    "Free parking on street", "Game console", "Garden or backyard",
    "Grab-rails for shower and toilet", "Ground floor access", "Gym", "Hair dryer",
    "Hand or paper towel", "Hand soap", "Handheld shower head", "Hangers", "Heating",
    "High chair", "Host greets you", "Hot tub", "Hot water", "Hot water kettle",
    "Indoor fireplace", "Internet", "Iron", "Keypad", "Kitchen", "Lake access",
    "Laptop friendly workspace", "Lock on bedroom door", "Lockbox",
    "Long term stays allowed", "Luggage dropoff allowed", "Microwave", "Other",
    "Other pet(s)", "Outlet covers", "Oven", "Pack ’n Play/travel crib",
    "Paid parking off premises", "Path to entrance lit at night", "Patio or balcony",
    "Pets allowed", "Pets live on this property", "Pocket wifi", "Pool",
    "Private bathroom", "Private entrance", "Private living room", "Refrigerator",
    "Roll-in shower with chair", "Room-darkening shades", "Safety card",
    "Self Check-In", "Shampoo", "Single level home", "Ski in/Ski out", "Smart lock",
    "Smartlock", "Smoke detector", "Smoking allowed", "Stair gates",
    "Step-free access", "Stove", "Suitable for events", "TV", "Table corner guards",
    "Toilet paper", "Washer", "Washer / Dryer", "Waterfront",
    "Well-lit path to entrance", "Wheelchair accessible", "Wide clearance to bed",
    "Wide clearance to shower and toilet", "Wide doorway", "Wide entryway",
    "Wide hallway clearance", "Window guards", "Wireless Internet",
    "hosting_amenity_49", "hosting_amenity_50"
]

In [48]:
print(len(amenities_columns))
print(len(set(amenities_columns)))
print(len(amenities_list))

129
129
122


In [49]:
# compara amenities_list y amenities_columns y dame una lista de los que no estan en amenities_columns
missing_amenities = list(set(amenities_columns) - set(amenities_list))

In [50]:
missing_amenities

['hosting_amenity_49',
 'hosting_amenity_50',
 'Paid parking off premises',
 'Other',
 'Long term stays allowed',
 'Flat',
 'Washer',
 'Washer / Dryer',
 'Smoking allowed',
 'Luggage dropoff allowed',
 'smooth pathway to front door',
 'Firm matress']

In [51]:
df.columns

Index(['log_price', 'amenities', 'accommodates', 'bathrooms', 'cleaning_fee',
       'host_has_profile_pic', 'host_identity_verified', 'host_response_rate',
       'instant_bookable', 'latitude', 'longitude', 'number_of_reviews',
       'review_scores_rating', 'bedrooms', 'beds',
       'property_type_Bed & Breakfast', 'property_type_Boat',
       'property_type_Boutique hotel', 'property_type_Bungalow',
       'property_type_Cabin', 'property_type_Camper/RV',
       'property_type_Casa particular', 'property_type_Castle',
       'property_type_Cave', 'property_type_Chalet',
       'property_type_Condominium', 'property_type_Dorm',
       'property_type_Earth House', 'property_type_Guest suite',
       'property_type_Guesthouse', 'property_type_Hostel',
       'property_type_House', 'property_type_Hut', 'property_type_In-law',
       'property_type_Island', 'property_type_Lighthouse',
       'property_type_Loft', 'property_type_Other',
       'property_type_Parking Space', 'property_ty

In [52]:
df = df.drop(columns=['amenities', 'clean_amenities'])

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59463 entries, 0 to 59462
Data columns (total 74 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   log_price                            59463 non-null  float64
 1   accommodates                         59463 non-null  int64  
 2   bathrooms                            59463 non-null  float64
 3   cleaning_fee                         59463 non-null  int64  
 4   host_has_profile_pic                 59463 non-null  int64  
 5   host_identity_verified               59463 non-null  int64  
 6   host_response_rate                   59463 non-null  float64
 7   instant_bookable                     59463 non-null  int64  
 8   latitude                             59463 non-null  float64
 9   longitude                            59463 non-null  float64
 10  number_of_reviews                    59463 non-null  int64  
 11  review_scores_rating        

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59463 entries, 0 to 59462
Data columns (total 74 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   log_price                            59463 non-null  float64
 1   accommodates                         59463 non-null  int64  
 2   bathrooms                            59463 non-null  float64
 3   cleaning_fee                         59463 non-null  int64  
 4   host_has_profile_pic                 59463 non-null  int64  
 5   host_identity_verified               59463 non-null  int64  
 6   host_response_rate                   59463 non-null  float64
 7   instant_bookable                     59463 non-null  int64  
 8   latitude                             59463 non-null  float64
 9   longitude                            59463 non-null  float64
 10  number_of_reviews                    59463 non-null  int64  
 11  review_scores_rating        

In [55]:
# Normaliza latitud y longitud
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[['latitude', 'longitude']] = scaler.fit_transform(df[['latitude', 'longitude']])
df[['latitude', 'longitude']].describe()

,latitude,longitude
count,59463.000000,59463.000000
mean,0.574471,0.601105
std,0.337538,0.418379
min,0.000000,0.000000
25%,0.089185,0.081215
50%,0.810389,0.941291
75%,0.818465,0.942397
max,1.000000,1.000000


In [56]:
# Crea una función que vea el Dtype de cada columna y si es bool lo cambie a int
def convertir_bool_a_int(df):
    for column in df.columns:
        if df[column].dtype == 'bool':
            df[column] = df[column].astype(int)
    return df

In [57]:
df = convertir_bool_a_int(df)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59463 entries, 0 to 59462
Data columns (total 74 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   log_price                            59463 non-null  float64
 1   accommodates                         59463 non-null  int64  
 2   bathrooms                            59463 non-null  float64
 3   cleaning_fee                         59463 non-null  int64  
 4   host_has_profile_pic                 59463 non-null  int64  
 5   host_identity_verified               59463 non-null  int64  
 6   host_response_rate                   59463 non-null  float64
 7   instant_bookable                     59463 non-null  int64  
 8   latitude                             59463 non-null  float64
 9   longitude                            59463 non-null  float64
 10  number_of_reviews                    59463 non-null  int64  
 11  review_scores_rating        

In [59]:
df.to_csv('../data/ml/Airbnb_Data_Cleaned.csv', index=False)